In [1]:
# Application Description:
# FPGA-based hardware accelerator for Google's MLP-Mixer model.
# *** Software-only implementation for testing purposes ***

# Instructions:
# 1. Ensure you have the libraries installed (pip install timm torch pillow)
# 2. Run the jupyter notebook cells in order
# 3. Feel free to change the model name to any timm MLP-Mixer model
# 4. Feel free to change the input image url to any image of your choice

# Performance:
# For the current single-image test, the bulk of the inference time is spent looping through the model layers
# MLP-Mixer has no attention mechanism - all mixing is done via MLPs over tokens and channels

In [2]:
"""
Load the pre-trained MLP-Mixer model
"""

from urllib.request import urlopen
from PIL import Image
import timm
import torch
import torch.nn.functional as F

model_name = 'mixer_b16_224.goog_in21k_ft_in1k'

model = timm.create_model(model_name, pretrained=True)
model = model.eval()

stem = model.stem # Patch embedding (Conv2d projecting patches -> tokens)
blocks = model.blocks # List of MixerBlock layers
norm = model.norm # Final layer norm
head = model.head # Classification head (Linear)

# Get model-specific transforms (resize to 224x224, normalization, etc.)
data_config = timm.data.resolve_model_data_config(model)
transforms = timm.data.create_transform(**data_config, is_training=False)

print(f"Loaded model: {model_name}")
print(f"Num mixer blocks: {len(blocks)}")
print(f"Hidden dim (num_features): {model.num_features}")

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded model: mixer_b16_224.goog_in21k_ft_in1k
Num mixer blocks: 12
Hidden dim (num_features): 768


In [3]:
"""
Software Step: Process image(s) for input
"""

# Load and convert image
url = 'https://img.freepik.com/free-vector/realistic-blue-umbrella_1284-11412.jpg'
image = Image.open(urlopen(url)).convert("RGB")
input_tensor = transforms(image).unsqueeze(0)  # [1, 3, 224, 224]
print(f"Input tensor shape: {input_tensor.shape}")

# Patch embedding (stem) for input tokens
with torch.no_grad():
    x = stem(input_tensor)
    print(f"Shape after patch embedding (stem): {x.shape}")
    # Output: [B, num_patches, hidden_dim] = [1, 196, 768]

Input tensor shape: torch.Size([1, 3, 224, 224])
Shape after patch embedding (stem): torch.Size([1, 196, 768])


In [4]:
"""
Hardware Step: MLP-Mixer Blocks
Each block has two MLP sub-layers:
    (a) Token-mixing MLP  — mixes information across the 196 spatial tokens (transposed input)
    (b) Channel-mixing MLP — mixes information across the 768 channel dimensions
"""

with torch.no_grad():

    for i, block in enumerate(blocks):

        # (a) Token-Mixing MLP Block

        # Layer norm over channels before token mixing
        normed_tokens = block.norm1(x) # [B, num_patches, hidden_dim]

        # Transpose so MLP operates across the token (spatial) dimension
        normed_tokens_T = normed_tokens.transpose(1, 2) # [B, hidden_dim, num_patches]

        # Token-mixing MLP weights & biases (FPGA static weights)
        tok_fc1_w = block.mlp_tokens.fc1.weight.detach()
        tok_fc1_b = block.mlp_tokens.fc1.bias.detach()
        tok_fc2_w = block.mlp_tokens.fc2.weight.detach()
        tok_fc2_b = block.mlp_tokens.fc2.bias.detach()

        # Two-layer MLP with GELU activation
        tok_hidden = F.gelu(torch.matmul(normed_tokens_T, tok_fc1_w.T) + tok_fc1_b)
        tok_out_T  = torch.matmul(tok_hidden, tok_fc2_w.T) + tok_fc2_b

        # Transpose back and apply residual connection
        x = x + tok_out_T.transpose(1, 2) # [B, num_patches, hidden_dim]

        #----------------------------------------------------------------------------------------------#

        # (b) Channel-Mixing MLP Block

        # Layer norm over channels before channel mixing
        normed_channels = block.norm2(x) # [B, num_patches, hidden_dim]

        # Channel-mixing MLP weights & biases (FPGA static weights)
        ch_fc1_w = block.mlp_channels.fc1.weight.detach()
        ch_fc1_b = block.mlp_channels.fc1.bias.detach()
        ch_fc2_w = block.mlp_channels.fc2.weight.detach()
        ch_fc2_b = block.mlp_channels.fc2.bias.detach()

        # Two-layer MLP with GELU activation
        ch_hidden = F.gelu(torch.matmul(normed_channels, ch_fc1_w.T) + ch_fc1_b)
        ch_out    = torch.matmul(ch_hidden, ch_fc2_w.T) + ch_fc2_b

        # Residual connection
        x = x + ch_out # [B, num_patches, hidden_dim]

In [ ]:
"""
Software Step: Final feature embedding
"""

with torch.no_grad():
    
    # Final layer norm
    # Equivalent to model.forward_features(transforms(image).unsqueeze(0))
    x = norm(x) # [B, num_patches, hidden_dim]

    # Global average pool over 196 tokens
    # Equivalent to model.forward_head(x, pre_logits=True)
    feature_embedding = x.mean(dim=1) # [B, hidden_dim]
    print(f"Shape of feature embedding: {feature_embedding.shape}")

Shape of feature embedding: torch.Size([1, 768])


In [6]:
"""
Optional: Classification
"""

ENABLE_CLASSIFICATION = True

if ENABLE_CLASSIFICATION:
    with torch.no_grad():
        # [1, 1000]
        logits = head(feature_embedding)
        pred_idx = logits.argmax(-1).item()

    from timm.data.imagenet_info import ImageNetInfo
    # 1000 classes
    imagenet_info = ImageNetInfo()
    label_descriptions = imagenet_info.label_descriptions(detailed=True, as_dict=False)

    pred_label = label_descriptions[pred_idx]
    print(f"Predicted class - {pred_label} (ID: {pred_idx})")

Predicted class - umbrella: a lightweight handheld collapsible canopy (ID: 879)
